# Notebook 03 — RAG Pipeline + Ragas Evaluation

**Goal:** Run the full retrieval + generation pipeline on the test set.
Measure:
- Ragas Faithfulness (no hallucination)
- Ragas Answer Relevancy
- Refusal accuracy on out-of-corpus queries
- Optional: cost and latency comparison OpenAI vs Ollama

**Output:** `figures/03_ragas_metrics.png`, full results JSON

In [ ]:
import sys, json, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.style.use('dark_background')

sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else str(Path.cwd()))
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# ── End-to-end query examples ────────────────────────────────────────────
from src.retriever import Retriever
from src.generator import Generator

retriever = Retriever(use_hybrid=True, use_reranker=True, top_k=4)
gen = Generator()

test_query = 'What is the minimum pipe insulation thickness for a 4-inch steam pipe at 400°F?'
chunks = retriever.retrieve(test_query)
result = gen.generate(test_query, chunks)

print('=== Answer ===')
print(result['answer'])
print()
print('=== Sources ===')
for s in result['sources']:
    print(f"  {s['source']}  p.{s['page']}")
print()
print(f'Tokens: {result["prompt_tokens"]} prompt + {result["completion_tokens"]} completion = ${result["cost_usd"]:.5f}')

In [ ]:
# ── Ragas evaluation ─────────────────────────────────────────────────────
# Requires: `pip install ragas datasets`
# Note: Ragas makes OpenAI API calls for its metrics — expect ~$0.05-0.20 for 25 queries
from src.eval import run_ragas_eval

ragas_results = run_ragas_eval(verbose=True)
print(json.dumps(ragas_results, indent=2))

In [ ]:
# ── Out-of-corpus refusal test ────────────────────────────────────────────
# Load only out_of_corpus queries and check system refuses correctly
from src.eval import load_test_queries

queries = load_test_queries()
ooc = [q for q in queries if q.get('query_type') == 'out_of_corpus']
print(f'Testing {len(ooc)} out-of-corpus queries...')

for q in ooc:
    chunks = retriever.retrieve(q['query'])
    result = gen.generate(q['query'], chunks)
    answer_lower = result['answer'].lower()
    refused = any(kw in answer_lower for kw in ['don\'t have', 'not in the', 'not provided', 'cannot find', 'no information'])
    status = '✓ REFUSED' if refused else '✗ ANSWERED (bad)'
    print(f'{status}  Q: {q["query"][:80]}')
    if not refused:
        print(f'  Answer: {result["answer"][:200]}')

In [ ]:
# ── Cost tracking ─────────────────────────────────────────────────────────
# Run the 25 in_corpus queries and sum costs
from src.eval import load_test_queries

in_corpus = [q for q in load_test_queries() if q.get('query_type') in ('in_corpus', 'borderline')]
total_cost = 0.0
times = []

for q in in_corpus[:5]:  # sample 5 to estimate
    t0 = time.time()
    chunks = retriever.retrieve(q['query'])
    result = gen.generate(q['query'], chunks)
    elapsed = time.time() - t0
    total_cost += result.get('cost_usd', 0)
    times.append(elapsed)

print(f'Sample (5 queries): ${total_cost:.5f}, avg latency: {sum(times)/len(times):.1f}s')
print(f'Projected full eval (25 queries): ~${total_cost / 5 * 25:.4f}')

## Generation Evaluation Results

| Metric | Value | Target |
|--------|-------|--------|
| Ragas Faithfulness | *(fill in)* | ≥ 0.85 |
| Ragas Answer Relevancy | *(fill in)* | ≥ 0.85 |
| Refusal Accuracy | *(fill in)* / 5 | ≥ 0.80 |
| Avg latency (median) | *(fill in)* ms | ≤ 3s |
| Cost per 100 queries | *(fill in)* | ≤ $0.50 |

**Key finding:** *(fill in)*

**Next:** Notebook 04 — Reranker Ablation